In [0]:

-- Name:       Tanya Konono
-- Project:    CASE STUDY 1 - Coffee Sales Analysis
------------------------------------------------------------------------------

-- CHECKS for Null Values, Column Quality, Missing Values was done in POWER BI 

-- Overview of the table
Select * from transactions limit 50;

-- Views are 
Create VIEW MainTable as 
Select  transaction_id, 
        transaction_date,
        transaction_time,
        Case when hour(transaction_time) = 6 then '6-7am' --Add Time Brackets
             when hour(transaction_time) = 7 then '7-8am'
             when hour(transaction_time) = 8 then '8-9am'
             when hour(transaction_time) = 9 then '8-9am'
             when hour(transaction_time) = 10 then '9-10am'
             when hour(transaction_time) = 11 then '11am-12'
             when hour(transaction_time) = 12 then 'noon'
             when hour(transaction_time) = 13 then '1-2pm'
             when hour(transaction_time) = 14 then '2-3pm'
             when hour(transaction_time) = 15 then '3-4pm'
             when hour(transaction_time) = 16 then '4-5pm'
             when hour(transaction_time) = 17 then '5-6pm'
             when hour(transaction_time) = 18 then '6-7pm'
             when hour(transaction_time) = 19 then '7-8pm'
             when hour(transaction_time) = 20 then '8-9pm'
        End AS Time_Bracket,
        transaction_qty,
        store_location,
        product_category,
        product_type,
        product_detail,
        -- product_id and store_id are Not needed for this analysis
        unit_price * transaction_qty as Revenue
from transactions;

Select * from maintable;

---------------------------------------------------------------------------------------
-- CHECKING THAT ALL ENTRIES ARE INCLUDED IN MY TIME BRACKETS:
--------------------------------------------------------------------------------------- 
-- We know that there are 149116 records in this data-set
-- We use a CTE to Calculate the Total Number of Transactions over all time brackets.


With 
Known_figure as(
    Select Count (transaction_id) as Actual_Figure 
    from maintable
),
Calculated_Figure as(
    With Transactions_vs_TimeBracket as 
        (
            Select  time_bracket, 
                    count(transaction_id) as Hour_Transactions
            From maintable
            group by time_bracket
        )
    Select sum (Hour_Transactions) as Calculated_Figure 
    from Transactions_vs_TimeBracket
)
Select * from known_figure, Calculated_Figure;


--------------------------------------------------
-- Sales Per Month 
--------------------------------------------------



-- Desctripitve Statistics
With Revenue as (
      Select 
        transaction_qty * unit_price
    From transactions
)
Select  
    Count(*) as total_transactions,
    Round(Avg(Revenue),2) as Mean_Revenue,
    Round(Stddev(Revenue),2) as Stddev_Revenue,
    Round(variance(Revenue),2) as variance,
    Round(Min(Revenue),2) as Min_Revenue,
    Round(Max(Revenue),2) as Max_Revenue,
    Round(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY Revenue),2) as Median_Revenue,
    Max(Revenue) - Min(Revenue) as Range_Revenue
From transactions;

-- Percentiles and Quartiles
Select 
    ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY Revenue),2) as Q1,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY Revenue),2) as Q3,
    Round(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY Revenue),2) as Q2_Median,
    Round(PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY Revenue),2) as P10,
    Round(PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY Revenue), as P90
From transactions;

-- Outlier Detection - IQR Method
With Quartiles as (
  Select
        ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY Revenue),2) as Q1,
        ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY Revenue),2) as Q3
        From transactions
), 
IQR_Calculation as (
  Select 
      q1, q3, (q3-q1) as IQR
  From Quartiles
)
Select COUNT(Case when Revenue < q1 - 1.5*iqr then 1 end) as Lower_Outliers,
       COUNT(Case when Revenue > q3 + 1.5*iqr then 1 end) as Upper_Outliers, -- we can then add Lower_Outliers + Upper_Outliers as Total_Outliers
       ROUND (AVG(CASE WHEN Revenue >= (q1 - 1.5*iqr) and Revenue <= (q3 - 1.5*iqr) THEN Revenue END) as Mean_without_Outliers
From transactions, IQR_Calculation;

-- Outlier Detection - Z-Score Method
WITH stats AS (
    SELECT 
        AVG(Revenue) AS mean,
        STDDEV(Revenue) AS stddev
    FROM transactions
)
SELECT 
    transaction_id, Revenue, 
    Round((Revenue - mean)/stddev,2) as z_score,
CASE  
    WHEN ABS((Revenue - mean)/stddev) > 3 then 'Extreme / Outlier'
    WHEN ABS((Revenue - mean)/stddev) > 2 then 'Moderate'
    ELSE 'Normal'
END AS Anomaly_Flag
FROM transactions, stats;

-- Check Times
Select 
  transaction_date,
  max(transaction_time) as closing_time,
  min(transaction_time) as opening_time
 from transactions
 group by transaction_date
 Limit 7

-- checking the transaction times to ascertain what time this shop opens and closes
 Select transaction_date,transaction_time from transactions where transaction_date = '2023-01-01'

Select Count(transaction_id) as total_transactions
 from transactions

with transactions_by_time as (
 SELECT Count(transaction_qty) as totals1,
        Case when hour(transaction_time) between 7 and 10 then 'Morning rush'
            when hour(transaction_time) between 11 and 12 then 'brunch'
            when hour(transaction_time) between 13 and 15 then 'daytime'
            when hour(transaction_time) between 16 and 18 then 'afternoon'
            else 'evening' 
        End AS Time_Buckets
 from transactions
 Group by Time_Buckets
)
select sum(totals1) from transactions_by_time
--149116 total transactions